# **Fama–French 5-Factor Model**
### **Factors:** `MKT` | `SMB` | `HML` | `RMW` | `CMA`

---

### **Overview**
- The **Fama–French 5-factor model** explains stock returns using **five systematic risk factors**
- It extends:
  - **CAPM**
  - **Fama–French 3-factor model**
- Adds:
  - **Profitability**
  - **Investment behavior**

---

### **Model Equation**
- **`Ri - Rf = α + βm(MKT) + βs(SMB) + βh(HML) + βr(RMW) + βc(CMA) + ε`**

---

### **Terms Explanation**
- **`Ri`** → Return of asset
- **`Rf`** → Risk-free rate
- **`MKT`** → Market return minus risk-free rate
- **`α` (Alpha)** → Excess return
- **`ε`** → Error term

---

### **Factor Definitions**

#### **1. `MKT` (Market Risk Premium)**
- **`MKT = Rm - Rf`**
- Captures overall market movement

---

#### **2. `SMB` (Small Minus Big)**
- Small-cap returns − Large-cap returns
- Proxy: Market capitalization

---

#### **3. `HML` (High Minus Low)**
- Value stocks − Growth stocks
- Proxy: Book-to-market ratio

---

#### **4. `RMW` (Robust Minus Weak)**
- High profitability − Low profitability
- Proxy: (Revenue − COGS − SG&A − Interest) / Book equity

---

#### **5. `CMA` (Conservative Minus Aggressive)**
- Low investment − High investment
- Proxy: Total asset growth from FY(t-1) to FY(t)

---

### **Intuition**
- Returns are driven by **systematic factor exposure**
- Each stock has **factor sensitivities (`β`)**
- These determine expected return

---

### **Why This Model Matters**
- Explains cross-sectional return differences
- Used in:
  - Portfolio construction
  - Quant investing
  - Asset pricing

---

### **Limitations**
- Linear assumptions
- Static factors
- Missing:
  - `Momentum`
  - Nonlinear effects

---

### **Project Connection**
- Construct all 5 factors from raw data
- Use them to:
  - Explain returns
  - Estimate alpha (pricing error)
- Extend with ML

---

### **Expected Outputs**
- Factor series: `MKT`, `SMB`, `HML`, `RMW`, `CMA`
- Factor loadings
- Alpha
- Backtest results

### Phase 1: Data ingestion and Cleaning (Nifty 500: From 01-01-2011 till date)

In [2]:
# ==========================================
# Phase 1: Data Collection (NIFTY 500)
# ==========================================

from nselib import capital_market
import pandas as pd
from datetime import date, datetime, timedelta
import time
import os
import requests
import shutil
from pathlib import Path
import numpy as np 


In [3]:
# import sys
# !{sys.executable} -m pip install nselib
# !{sys.executable} -m pip install feedparser

In [4]:
# =====================================
# STEP 0: Project Setup
# =====================================

RUN_DATE          = date.today().isoformat()
FORCE_REFETCH     = False          # True = ignore price cache, re-download everything
SCREENER_SNAPSHOT = "2026-10-14"   # reuse the cached Screener pages (folder in data/raw/screener_html/)
CLEAN_DERIVED     = False          # True ONLY when rebuilding ALL of Phase 1 from raw data (wipes derived folders)

PROJECT_ROOT = Path().resolve().parent
DATA_DIR     = PROJECT_ROOT / "data"

# ---- RAW: exactly as received. Code never deletes these. ----
RAW_DIR      = DATA_DIR / "raw"
UNIVERSE_DIR = RAW_DIR / "universe"
PRICE_DIR    = RAW_DIR / "prices"
SCREENER_DIR = RAW_DIR / "screener_html" / SCREENER_SNAPSHOT
RF_DIR       = RAW_DIR / "rf"
MANIFEST_DIR = DATA_DIR / "manifests"

# ---- DERIVED: rebuilt from raw ----
INTERIM_DIR       = DATA_DIR / "interim"
PROCESSED_DIR     = DATA_DIR / "processed"
FIN_PROCESSED_DIR = DATA_DIR / "fin_data_processed"
FIN_BACKUPS_DIR   = INTERIM_DIR / "fin_data_backups"

if CLEAN_DERIVED:
    for d in [INTERIM_DIR, PROCESSED_DIR, FIN_PROCESSED_DIR]:
        if d.exists():
            shutil.rmtree(d)                  # only derived folders

for d in [UNIVERSE_DIR, PRICE_DIR, SCREENER_DIR, RF_DIR, MANIFEST_DIR,
          INTERIM_DIR, PROCESSED_DIR, FIN_PROCESSED_DIR, FIN_BACKUPS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ---- All run parameters in one place ----
START_DATE  = date(2011, 1, 1)
END_DATE    = date.today().replace(day=1) - timedelta(days=1)   # last day of the last COMPLETE month
LAG_MONTHS  = 3                     # months from fiscal year-end until the numbers are public
KEEP_SERIES = ["EQ", "BE", "BZ"]    # BE/BZ = trade-to-trade (often distressed stocks), must be kept
SERIES_RANK = {"EQ": 0, "BE": 1, "BZ": 2}   # if a stock trades in two series on a day, keep EQ

CA_DIR   = RAW_DIR / "corporate_actions"   # NSE corporate actions (dividends, splits, bonuses, demergers, rights)
BHAV_DIR = RAW_DIR / "bhavcopy"            # NSE daily bhavcopy (whole market)
CA_DIR.mkdir(parents=True, exist_ok=True)
BHAV_DIR.mkdir(parents=True, exist_ok=True)

# Remove half-written files left by a crashed run (never real data)
for junk in PRICE_DIR.glob("*.tmp"):
    junk.unlink()


def clean_cols(cols):
    """Fixes the BOM/quote garbage like 'ã¯â»â¿"symbol"' at source."""
    return (pd.Index(cols)
            .str.replace("\ufeff", "", regex=False)
            .str.replace("ï»¿", "", regex=False)
            .str.replace('"', "", regex=False)
            .str.strip().str.lower())


def to_number(col):
    """'1,234.50' -> 1234.5"""
    return pd.to_numeric(col.astype(str).str.replace(",", ""), errors="coerce")


print("Ready. Cached price files:", len(list(PRICE_DIR.glob("*.csv"))),
      "| Screener pages:", len(list(SCREENER_DIR.glob("*.html"))),
      "| CLEAN_DERIVED:", CLEAN_DERIVED)

Ready. Cached price files: 528 | Screener pages: 990 | CLEAN_DERIVED: False


In [5]:
# Month-end code: "ME" in pandas >= 2.2, "M" in older versions (same meaning)
pd_major, pd_minor = (int(x) for x in pd.__version__.split(".")[:2])
MONTH_END = "ME" if (pd_major, pd_minor) >= (2, 2) else "M"
print("pandas", pd.__version__, "-> month-end code:", MONTH_END)

pandas 2.1.4 -> month-end code: M


In [6]:
# # ONE-TIME: move old data/raw/*.csv  ->  data/raw/prices/
# old_files = list(RAW_DIR.glob("*.csv"))
# for f in old_files:
#     df = pd.read_csv(f, encoding="utf-8")
#     df.columns = clean_cols(df.columns)
#     df = df.rename(columns={"symbol": "nse_symbol"})   # NSE's historical symbol, e.g. SRTRANSFIN
#     df["date"] = pd.to_datetime(df["date"], dayfirst=True, errors="coerce")
#     df = df.dropna(subset=["date"]).drop_duplicates(["date", "series"]).sort_values("date")
#     df.to_csv(PRICE_DIR / f.name, index=False, date_format="%Y-%m-%d")
#     f.unlink()
# print("Migrated", len(old_files), "files")

In [7]:
# =======================================================================================================================
# STEP 1: NIFTY 500 tickers (frozen list of 14-Oct-2026; Stage 2 replaces it with a point-in-time universe)
# =======================================================================================================================

UNIVERSE_FILE = UNIVERSE_DIR / "nifty500_2026-10-14.csv"
df_list = pd.read_csv(UNIVERSE_FILE)
df_list = df_list[~df_list["Symbol"].str.upper().str.startswith("DUMMY")].copy()
tickers = df_list["Symbol"].tolist()
print(f"Total tickers: {len(tickers)}")

# Banks, NBFCs, insurers, AMCs: FF-style operating profitability is not defined for them
FIN_SYMBOLS = set(df_list.loc[df_list["Industry"] == "Financial Services", "Symbol"])
print("Financial firms in list:", len(FIN_SYMBOLS))

Total tickers: 500
Financial firms in list: 98


In [8]:
df_list.head()

,Company Name,Industry,Symbol,Series,ISIN Code
0,360 ONE WAM Ltd.,Financial Services,360ONE,EQ,INE466L01038
1,ABB India Ltd.,Capital Goods,ABB,EQ,INE117A01022
2,ACC Ltd.,Construction Materials,ACC,EQ,INE012A01025
3,ACME Solar Holdings Ltd.,Power,ACMESOLAR,EQ,INE622W01025
4,AIA Engineering Ltd.,Capital Goods,AIAENG,EQ,INE212H01026


In [9]:
df_list.shape

(500, 5)

In [10]:
# =====================================================================
# STEP 1b: Download NSE daily bhavcopies (ALL stocks, incl. later delisted)
# =====================================================================
from nselib.libutil import nse_urlfetch

UDIFF_START = date(2024, 7, 8)     # NSE changed the bhavcopy format on this date

def bhavcopy_url(d):
    if d < UDIFF_START:
        mon = d.strftime("%b").upper()                     # e.g. "JAN"
        return (f"https://nsearchives.nseindia.com/content/historical/EQUITIES/"
                f"{d.year}/{mon}/cm{d:%d}{mon}{d.year}bhav.csv.zip")
    return (f"https://nsearchives.nseindia.com/content/cm/"
            f"BhavCopy_NSE_CM_0_0_0_{d:%Y%m%d}_F_0000.csv.zip")

def fetch_bhavcopy(d):
    path = BHAV_DIR / f"{d:%Y-%m-%d}.csv.zip"
    if path.exists():
        return "cached"
    try:
        r = nse_urlfetch(bhavcopy_url(d))
    except Exception as e:
        return f"error: {e}"
    if r.status_code != 200 or len(r.content) < 1000:
        return "no_file"                                   # market holiday
    path.write_bytes(r.content)
    time.sleep(0.5)
    return "saved"

bhav_log = pd.DataFrame([{"date": d.date(), "status": fetch_bhavcopy(d.date())}
                         for d in pd.bdate_range(START_DATE, END_DATE)])
bhav_log.to_csv(MANIFEST_DIR / f"bhavcopy_{RUN_DATE}.csv", index=False)
print(bhav_log["status"].str.split(":").str[0].value_counts())

status
cached     3905
no_file     225
Name: count, dtype: int64


In [ ]:
# ==================================
# STEP 2: Fetch prices (incremental cache)
# ==================================
def _atomic_write_csv(df, path):
    tmp = path.with_suffix(".tmp")
    df.to_csv(tmp, index=False, date_format="%Y-%m-%d")
    os.replace(tmp, path)                     # all-or-nothing

def fetch_symbol_prices(symbol, start, end, force=False, retries=3):
    path, old, fetch_from = PRICE_DIR / f"{symbol}.csv", None, start

    if path.exists() and not force:
        old = pd.read_csv(path, parse_dates=["date"])
        if not old.empty:
            if date.fromtimestamp(path.stat().st_mtime) == date.today():
                return "cached", old          # already checked today
            fetch_from = old["date"].max().date() + timedelta(days=1)
            if fetch_from > end:
                return "cached", old

    new = None
    for attempt in range(1, retries + 1):
        try:
            new = capital_market.price_volume_and_deliverable_position_data(
                symbol=symbol,
                from_date=fetch_from.strftime("%d-%m-%Y"),
                to_date=end.strftime("%d-%m-%Y"))
            break
        except Exception as e:
            if attempt == retries:
                return f"error: {e}", old
            time.sleep(2 ** attempt)          # 2s, 4s, 8s

    if new is None or new.empty:
        if old is not None:
            path.touch()                      # mark "checked today"
            return "no_new_rows", old
        return "no_data", None

    new.columns = clean_cols(new.columns)
    new = new.rename(columns={"symbol": "nse_symbol"})
    new["date"] = pd.to_datetime(new["date"], dayfirst=True, errors="coerce")
    new = new.dropna(subset=["date"])
    # Raw keeps ALL series (EQ/BE/BZ); filtering happens in Cell [11]

    combined = new if old is None else pd.concat([old, new], ignore_index=True)
    combined = (combined.drop_duplicates(["date", "series"], keep="last")
                        .sort_values("date"))
    _atomic_write_csv(combined, path)
    return ("updated" if old is not None else "new"), combined


def fetch_all_data(symbols, start, end, force=False):
    log = []
    
    for i, sym in enumerate(symbols, 1):
        status, df = fetch_symbol_prices(sym, start, end, force=force)
        has = df is not None and not df.empty
        log.append({"symbol": sym, "status": status,
                    "rows": len(df) if has else 0,
                    "first_date": df["date"].min().date() if has else None,
                    "last_date":  df["date"].max().date() if has else None,
                    "checked_on": RUN_DATE})
        print(f"[{i}/{len(symbols)}] {sym:<12} {status}")
        
        if status != "cached":
            time.sleep(1)                     # throttle only real network calls

    manifest = pd.DataFrame(log)
    manifest.to_csv(MANIFEST_DIR / f"prices_{RUN_DATE}.csv", index=False)
    manifest.to_csv(MANIFEST_DIR / "prices_latest.csv", index=False)
    
    print(manifest["status"].str.split(":").str[0].value_counts())
    
    return manifest

In [ ]:
# =======================================================================
# STEP 3: Execute for retreiving stock price data of 500 stocks from 2011
# =======================================================================

price_manifest = fetch_all_data(tickers, START_DATE, END_DATE, force=FORCE_REFETCH)
print("Symbols with data:", (price_manifest["rows"] > 0).sum())

In [ ]:
# =====================================================================
# STEP 3b: Dividends from NSE corporate actions
# =====================================================================

import re
for yr in range(START_DATE.year, END_DATE.year + 1):
    
    path = CA_DIR / f"corporate_actions_{yr}.csv"
    
    if path.exists() and yr < END_DATE.year:
        continue                                      # past years never change
    
    ca_year = capital_market.corporate_actions_for_equity(
        from_date=f"01-01-{yr}", to_date=f"31-12-{yr}")
    ca_year.to_csv(path, index=False)
    
    time.sleep(1)

ca = pd.concat([pd.read_csv(p) for p in sorted(CA_DIR.glob("corporate_actions_*.csv"))],
               ignore_index=True)
ca.columns = clean_cols(ca.columns)
print(ca.columns.tolist())        # expect: symbol, series, faceval, subject, exdate, ...

def dividend_per_share(subject, face_value):
    """
    'Interim Dividend - Rs 5 Per Share' -> 5.0 ; 'Dividend - 50%' with FV 10 -> 5.0
    Only the dividend part of a combined subject is read
    (e.g. '.../Dividend - Rs 10 Per Share/Face Value Split From Rs 10 To Re 1' -> 10, not 21).
    """
    parts = [p for p in str(subject).lower().split("/") if re.search(r"\bdiv(idend)?\b", p)]
    if not parts:                                     # no dividend (skips 'Sub-Division' splits)
        return 0.0
    text = " / ".join(parts)
    amounts = re.findall(r"(?:\brs|\bre|\binr|₹)\.?\s*-?\s*(\d+(?:\.\d+)?)", text)
    total = sum(float(a) for a in amounts)
    if total == 0:                                    # old style: dividend as % of face value
        pcts = re.findall(r"(\d+(?:\.\d+)?)\s*%", text)
        fv = pd.to_numeric(face_value, errors="coerce")
        if pcts and pd.notna(fv):
            total = sum(float(p) for p in pcts) / 100 * fv
    return total


ca["div_ps"]  = [dividend_per_share(s, fv) for s, fv in zip(ca["subject"], ca["faceval"])]
ca["ex_date"] = pd.to_datetime(ca["exdate"], dayfirst=True, errors="coerce")

divs = (ca[(ca["div_ps"] > 0) & ca["ex_date"].notna()]
        .groupby(["symbol", "ex_date"], as_index=False)["div_ps"].sum())

# lookup: (symbol on that day, ex-date) -> rupees per share
div_lookup = dict(zip(zip(divs["symbol"], divs["ex_date"]), divs["div_ps"]))

print("Dividend events:", len(divs), "| per year:")
print(divs["ex_date"].dt.year.value_counts().sort_index())

In [ ]:
# check 1
# What do split and bonus records look like in the NSE corporate-actions file?
ca_sb = ca[ca["subject"].str.contains("bonus|split|sub-div", case=False, na=False)]
print("Split/bonus records:", len(ca_sb))
for s in ca_sb["subject"].sample(25, random_state=1):
    print(s)

In [ ]:
# check 2
# Bhavcopy columns: old format and new (UDiFF) format. Is ISIN there?
bhav_files = sorted(BHAV_DIR.glob("*.zip"))
for f in [bhav_files[0], bhav_files[-1]]:
    print(f.name, pd.read_csv(f, nrows=2).columns.tolist())

### Step 3c: Corporate actions (splits, bonuses, demergers, rights issues)

**Why:** Step 4 assumed NSE's `prevclose` is adjusted on every ex-date. It is not (278 of 280 events checked were unadjusted), so raw returns contain fake crashes at splits/bonuses, and share counts backfilled with `ca_factor` never undid missed splits (ME overstated up to 10×).

**Sources and rules**
- **Splits / bonuses / consolidations** from the NSE corporate-actions file. Bonus X:Y → share count × (X+Y)/Y; split from Rs A to Rs B → × A/B.
- **Manual corrections** for events missing from the file or with wrong factors, found by the price-drop checks (price-implied factors).
- **Demergers:** the ex-day price drop is value moved to a new company, not a loss. The return on that day is neutralised in Step 4.
- **Rights issues:** return factor = cum price ÷ theoretical ex-rights price (CRSP convention). Partly-paid issues are treated as fully paid.
- Every event is mapped to the **first trading day on/after its ex-date**, matched on the NSE symbol used that day or today's symbol.

**Checks (all must PASS):** every split/bonus is confirmed by its price drop; every one-day move > 37.5% down or > 66% up is explained; every NSE-listed demerger of our stocks is in the list.

**Outputs:** `interim/split_bonus_events.csv`, `interim/demerger_days.csv`, `interim/rights_events.csv`

In [ ]:
# 3c.1 Parsers: share events (split / bonus / consolidation) and rights issues

import re

def parse_share_events(subject):
    """
    One corporate-action subject -> list of (event_type, share_mult).
    share_mult = new share count / old share count.
      'Bonus 1:2'                          -> [('bonus', 1.5)]    1 new for every 2 held
      'From Rs 10/- ... To Re 1/- ...'     -> [('split', 10.0)]   face value 10 -> 1
      'From Re 1 To Rs 10' (consolidation) -> [('split', 0.1)]
    """
    s = str(subject).lower()
    events = []

    # Bonus X:Y (skip bonus debentures / preference shares: not equity)
    for x, y in re.findall(r"bonus\s*(?:issue)?\s*(\d+)\s*:\s*(\d+)", s):
        if "debenture" in s or "preference" in s:
            continue
        x, y = float(x), float(y)
        if y > 0:
            events.append(("bonus", (x + y) / y))

    # Face value split / consolidation: "from rs 10 ... to re 1"
    if "split" in s or "sub-div" in s or "consolidat" in s:
        m = re.search(r"from\s*r[se]\.?\s*(\d+(?:\.\d+)?).*?to\s*r[se]\.?\s*(\d+(?:\.\d+)?)", s)
        if m:
            old_fv, new_fv = float(m.group(1)), float(m.group(2))
            if new_fv > 0 and old_fv != new_fv:
                events.append(("split", old_fv / new_fv))

    return events


def parse_rights(subject, face_value):
    """
    'Rights 8:15 @ Premium Rs 40/-' with face value 10 -> (8, 15, 50.0)
    Returns (new_shares, held_shares, issue_price) or None.
    Issue price = face value + premium (NSE quotes the premium over face value).
    """
    s = str(subject).lower()
    if "right" not in s or any(w in s for w in ["debenture", "ncd", "warrant", "preference"]):
        return None

    ratio = re.search(r"rights?\s*(?:issue)?[^0-9:]{0,40}?(\d+)\s*:\s*(\d+)", s)
    if ratio is None:
        return None
    new_sh, held_sh = float(ratio.group(1)), float(ratio.group(2))

    fv = pd.to_numeric(face_value, errors="coerce")
    if re.search(r"@\s*par\b", s) and pd.notna(fv):
        price = fv
    else:
        p = re.search(r"@\s*(premium\s*(?:of\s*)?)?r[se]\.?\s*(\d+(?:\.\d+)?)", s)
        if p is None:
            return None
        price = float(p.group(2))
        if p.group(1):                                  # quoted as a premium over face value
            if pd.isna(fv):
                return None
            price += fv

    if held_sh <= 0 or new_sh <= 0:
        return None
    return new_sh, held_sh, price


# Self-tests (every line must start with OK)
share_tests = {
    "Bonus 1:1": [("bonus", 2.0)],
    "Bonus 1:2": [("bonus", 1.5)],
    "Bonus 10:39": [("bonus", 1.2564)],
    "Face Value Split (Sub-Division) - From Rs 10/- Per Share To Re 1/- Per Share": [("split", 10.0)],
    "Face Value Split From Rs 10 To Rs 2": [("split", 5.0)],
    " Bonus 1:5/Face Value Split (Sub-Division) - From Rs 10/- Per Share To Rs 2/- Per Share":
        [("bonus", 1.2), ("split", 5.0)],
    "Annual General Meeting/Dividend Rs.3/- Per Share/Scheme Of Arrangement - Bonus Debentures 1:1 (Purpose Revised)": [],
    "Face Value Consolidation From Re 1/- Per Share To Rs 10/- Per Share": [("split", 0.1)],
    "Interim Dividend - Rs 5 Per Share": [],
}
for subject, expected in share_tests.items():
    got = [(t, round(f, 4)) for t, f in parse_share_events(subject)]
    print("OK  " if got == expected else "FAIL", got, "<-", subject[:60])

rights_tests = [
    ("Rights 8:15 @ Premium Rs 40/-", 10, (8.0, 15.0, 50.0)),
    ("Rights 1:15 @ Premium Rs 1247/-", 10, (1.0, 15.0, 1257.0)),
    ("Rights 2:7 @ Rs 205/-", 10, (2.0, 7.0, 205.0)),
    ("Rights 1:6 @ Par", 10, (1.0, 6.0, 10.0)),
    ("Rights 3:10 @ Premium Rs 20.50/- Per Share", 2, (3.0, 10.0, 22.5)),
    ("Rights Issue Of Non Convertible Debentures", 10, None),
    ("Dividend - Rs 5 Per Share", 10, None),
]
for subject, fv, expected in rights_tests:
    got = parse_rights(subject, fv)
    print("OK  " if got == expected else "FAIL", got, "<-", subject)

In [ ]:
# 3c.2 Event tables from the NSE corporate-actions file

# --- Splits / bonuses / consolidations: one row per stock per ex-date ---
rows = []
for sym, isin, ex_date, subject in zip(ca["symbol"], ca["isin"], ca["ex_date"], ca["subject"]):
    for event_type, mult in parse_share_events(subject):
        rows.append({"symbol": sym, "isin": isin, "ex_date": ex_date,
                     "event_type": event_type, "share_mult": mult})

events = pd.DataFrame(rows).dropna(subset=["ex_date"])
events = events.drop_duplicates(["symbol", "ex_date", "event_type", "share_mult"])   # "Purpose Revised" repeats

split_events = (events.groupby(["symbol", "ex_date"], as_index=False)
                      .agg(isin=("isin", "first"),
                           event_type=("event_type", lambda s: "+".join(sorted(s))),
                           share_mult=("share_mult", "prod")))       # bonus + split same day -> multiply
split_events["source"] = "nse_file"

# --- Rights issues: one row per stock per ex-date ---
rows = []
for sym, ex_date, subject, fv in zip(ca["symbol"], ca["ex_date"], ca["subject"], ca["faceval"]):
    parsed = parse_rights(subject, fv)
    if parsed is not None and pd.notna(ex_date):
        new_sh, held_sh, price = parsed
        rows.append({"symbol": sym, "ex_date": ex_date, "rights_new": new_sh,
                     "rights_held": held_sh, "issue_price": price, "subject": subject})

rights_events = pd.DataFrame(rows).drop_duplicates(["symbol", "ex_date"])

n_rights_records = ca["subject"].str.contains("right", case=False, na=False).sum()
print("Split/bonus events:", len(split_events), "| stocks:", split_events["symbol"].nunique())
print("Rights records in file:", n_rights_records, "| parsed:", len(rights_events))

In [ ]:
# 3c.3 Manual corrections (from checks 3c.5-3c.7 and the review of one-day falls)
# Symbol = NSE symbol used ON THAT DAY (today's symbol also works).

MANUAL_EVENTS = [
    # symbol,       ex_date,      share_mult   (price-implied: one-day price ratio, rounded)
    ("MANAPPURAM", "2011-06-09",  2.0),
    ("M&MFIN",     "2013-02-15",  5.0),
    ("ASIANPAINT", "2013-07-30", 10.0),
    ("SBIN",       "2014-11-20", 10.0),
    ("GRANULES",   "2015-03-23", 10.0),
    ("MOTHERSUMI", "2015-07-23",  1.5),     # bonus 1:2 (ratio 1.497)
    ("SOLARINDS",  "2016-07-13",  5.0),
    ("CAPLIPOINT", "2016-10-19",  5.0),
    ("KARURVYSYA", "2016-11-17",  5.0),
    ("CGCL",       "2016-11-23",  5.0),
    ("KPRMILL",    "2016-11-29",  2.0),
    ("JSWSTEEL",   "2017-01-04", 10.0),
    ("DAAWAT",     "2017-02-07", 10.0),     # now LTFOODS
    ("AJANTPHARM", "2022-06-22",  1.5),     # bonus 1:2 (ratio 1.515)
]

FIXED_FACTORS = [
    # symbol,      ex_date,      correct share_mult
    ("HINDZINC",  "2011-03-07", 10.0),      # file: split x5; same-day bonus 1:1 missing
    ("MINDACORP", "2015-01-05",  2.0),      # file: x10; price drop says x2
]

# Demergers: the ex-day price drop is value moved to a new company, NOT a loss.
# Ex-date as announced; it is mapped to the first trading day on/after it in 3c.4.
DEMERGER_DAYS = [
    ("ADANIENT",   "2015-06-03"),
    ("JSL",        "2015-12-04"),
    ("CROMPGREAV", "2016-03-15"),
    ("KPIT",       "2019-01-24"),
    ("IIFL",       "2019-06-14"),
    ("CENTURYTEX", "2019-10-11"),
    ("TATACHEM",   "2020-03-04"),
    ("FAIRCHEM",   "2020-08-21"),     # now PRIVISCL
    ("ANANTRAJ",   "2020-10-06"),     # first trade after it: 2020-10-20 (trading gap)
    ("GMRINFRA",   "2022-01-11"),     # now GMRAIRPORT
    ("MOTHERSUMI", "2022-01-14"),     # now MOTHERSON
    ("AARTIIND",   "2022-10-19"),
    ("NMDC",       "2022-10-27"),
    ("RELIANCE",   "2023-07-20"),     # Jio Financial
    ("ITC",        "2025-01-06"),     # ITC Hotels
    ("SIEMENS",    "2025-04-07"),
    ("STLTECH",    "2025-04-24"),
    ("ABFRL",      "2025-05-22"),
    ("TATAMOTORS", "2025-10-14"),
    ("HINDUNILVR", "2025-12-05"),
    ("VEDL",       "2026-04-30"),
    ("HEGAM",      "2026-09-07"),     # first trade after it: 2026-09-22 (trading gap)
]

# Big one-day falls checked and confirmed as real market moves (kept as they are)
REAL_MOVES = [
    ("YESBANK", "2020-03-06"),        # RBI moratorium
]

# 1. Fix wrong factors (each must change exactly one row)
for sym, dt, mult in FIXED_FACTORS:
    row = (split_events["symbol"] == sym) & (split_events["ex_date"] == pd.Timestamp(dt))
    print(f"Fix {sym} {dt}: rows matched = {row.sum()}")          # must be 1
    split_events.loc[row, ["share_mult", "source"]] = [mult, "nse_file, factor fixed"]

# 2. Add missing events (skip any the NSE file already has)
manual = pd.DataFrame(MANUAL_EVENTS, columns=["symbol", "ex_date", "share_mult"])
manual["ex_date"] = pd.to_datetime(manual["ex_date"])
manual["event_type"] = "manual"
manual["source"] = "manual (price-implied)"
split_events = (pd.concat([split_events, manual], ignore_index=True)
                  .drop_duplicates(["symbol", "ex_date"], keep="first")
                  .sort_values(["symbol", "ex_date"]).reset_index(drop=True))

demerger_days = pd.DataFrame(DEMERGER_DAYS, columns=["symbol", "ex_date"])
demerger_days["ex_date"] = pd.to_datetime(demerger_days["ex_date"])
real_moves = pd.DataFrame(REAL_MOVES, columns=["symbol", "ex_date"])
real_moves["ex_date"] = pd.to_datetime(real_moves["ex_date"])

print("\nSplit/bonus events after corrections:", len(split_events))
print(split_events["source"].value_counts())
print("Demergers:", len(demerger_days), "| real moves:", len(real_moves))

In [ ]:
# 3c.4 Daily prices of our stocks + map every event to its first trading day

daily_list = []
for f in PRICE_DIR.glob("*.csv"):
    dd = pd.read_csv(f, parse_dates=["date"])
    dd.columns = clean_cols(dd.columns)
    sym_col = "nse_symbol" if "nse_symbol" in dd.columns else "symbol"
    dd["series"] = dd["series"].astype(str).str.strip().str.upper()
    dd = dd[dd["series"].isin(KEEP_SERIES)].copy()
    dd["rank"] = dd["series"].map(SERIES_RANK)
    dd = dd.sort_values(["date", "rank"]).drop_duplicates("date", keep="first")
    dd["closeprice"] = to_number(dd["closeprice"])
    dd["yesterday_close"] = dd["closeprice"].shift(1)
    dd["gap_days"] = dd["date"].diff().dt.days            # long gap = no trading (suspension etc.)
    dd["file_symbol"] = f.stem
    daily_list.append(dd[["file_symbol", sym_col, "date", "closeprice", "yesterday_close", "gap_days"]]
                      .rename(columns={sym_col: "nse_symbol"}))

daily_all = pd.concat(daily_list, ignore_index=True).sort_values("date")
daily_all["price_jump"] = daily_all["yesterday_close"] / daily_all["closeprice"]   # >1 = price fell
LAST_PRICE_DATE = daily_all["date"].max()


def to_trading_day(table, max_days=30):
    """
    Attach file_symbol, trade_date (first trading day on/after ex_date) and that day's prices.
    Matches on the NSE symbol used that day AND on today's symbol, and keeps the EARLIER match
    (a renamed stock can match its new symbol only weeks later, e.g. MINDAIND -> UNOMINDA).
    max_days=30 also maps ex-dates that fall inside a trading gap.
    """
    left = table.sort_values("ex_date").reset_index(drop=True)
    kw = dict(left_on="ex_date", right_on="date", left_by="symbol",
              direction="forward", tolerance=pd.Timedelta(days=max_days))
    m1 = pd.merge_asof(left, daily_all, right_by="nse_symbol", **kw)
    m2 = pd.merge_asof(left, daily_all, right_by="file_symbol", **kw)
    use_m2 = m2["date"].notna() & (m1["date"].isna() | (m2["date"] < m1["date"]))
    m1.loc[use_m2, :] = m2.loc[use_m2, m1.columns]
    return m1.rename(columns={"date": "trade_date"})


# Demergers = manual list (3c.3) + every demerger in the NSE file (labelled reliably from ~2019)
dm_file = (ca[ca["subject"].str.contains("demerg", case=False, na=False)]
           .dropna(subset=["ex_date"]).drop_duplicates(["symbol", "ex_date"])[["symbol", "ex_date"]])
dm_file["source"] = "nse_file"
demerger_days["source"] = "manual"
demerger_all = to_trading_day(pd.concat([demerger_days, dm_file], ignore_index=True))
demerger_m = (demerger_all.dropna(subset=["trade_date"])
                          .sort_values("source")                       # "manual" before "nse_file"
                          .drop_duplicates(["file_symbol", "trade_date"], keep="first"))

split_m = to_trading_day(split_events)
real_m = to_trading_day(real_moves)
rights_m = to_trading_day(rights_events)

manual_unmapped = demerger_all[(demerger_all["source"] == "manual") & demerger_all["trade_date"].isna()]

print("Daily rows:", len(daily_all), "| stocks:", daily_all["file_symbol"].nunique(),
      "| last date:", LAST_PRICE_DATE.date())
print("Mapped to a trading day -> splits/bonuses:", split_m["trade_date"].notna().sum(),
      "| rights:", rights_m["trade_date"].notna().sum())
print("Demergers of our stocks:", len(demerger_m), "->", demerger_m["source"].value_counts().to_dict(),
      "| manual ones not mapped:", len(manual_unmapped))

In [ ]:
# 3c.5 CHECK 1: every split/bonus of our stocks is confirmed by the price drop on its ex-date
# NSE prevclose is not adjusted, so yesterday's close / today's close is close to share_mult.
# After a trading gap (> 5 days) the real move in between makes this test meaningless.

ours = split_m[split_m["trade_date"].notna()].copy()
ours["jump_vs_mult"] = ours["price_jump"] / ours["share_mult"]
ours["status"] = np.where(ours["gap_days"] > 5, "after trading gap",
                 np.where(ours["jump_vs_mult"].between(0.8, 1.25), "confirmed", "FAILED"))

future = split_m["trade_date"].isna() & (split_m["ex_date"] > LAST_PRICE_DATE)
print(ours["status"].value_counts())
print("Events after the last price date (ignored):", future.sum())
display(ours.loc[ours["status"] == "FAILED",
                 ["file_symbol", "ex_date", "trade_date", "share_mult", "price_jump", "jump_vs_mult"]])

In [ ]:
# 3c.6 CHECK 2: big one-day price moves (fall > 37.5% or rise > 66%) must all be explained

big = daily_all[(daily_all["price_jump"] > 1.6) | (daily_all["price_jump"] < 0.6)].copy()

split_keys = set(zip(split_m["file_symbol"], split_m["trade_date"]))
demerger_keys = set(zip(demerger_m["file_symbol"], demerger_m["trade_date"]))
real_keys = set(zip(real_m["file_symbol"], real_m["trade_date"]))
rights_keys = set(zip(rights_m["file_symbol"], rights_m["trade_date"]))

big_key = list(zip(big["file_symbol"], big["date"]))
big["explained_by"] = "UNKNOWN"
big.loc[big["gap_days"] > 30, "explained_by"] = "data gap (monthly return set to NaN)"
big.loc[[k in split_keys for k in big_key], "explained_by"] = "split/bonus"
big.loc[[k in demerger_keys for k in big_key], "explained_by"] = "demerger"
big.loc[[k in rights_keys for k in big_key], "explained_by"] = "rights issue"
big.loc[[k in real_keys for k in big_key], "explained_by"] = "real move"

print(big["explained_by"].value_counts())
display(big.loc[big["explained_by"] == "UNKNOWN",
                ["file_symbol", "nse_symbol", "date", "price_jump", "gap_days"]])

In [ ]:
# 3c.7 CHECK 3: demergers in the final list, by source (NSE-file ones are now added automatically)

print(demerger_m["source"].value_counts())
display(demerger_m.sort_values("trade_date")
        [["file_symbol", "nse_symbol", "ex_date", "trade_date", "price_jump", "gap_days", "source"]])
display(manual_unmapped)          # must be empty

In [ ]:
# 3c.8 Rights issues: return adjustment factor on the ex-date
# Theoretical ex-rights price TERP = (held x cum price + new x issue price) / (held + new)
# Return factor = cum price / TERP (> 1 only if the issue price is below the market price)

r = rights_m[rights_m["trade_date"].notna()].copy()
cum = r["yesterday_close"]
terp = (r["rights_held"] * cum + r["rights_new"] * r["issue_price"]) / (r["rights_held"] + r["rights_new"])
r["ret_factor"] = np.where(r["issue_price"] < cum, cum / terp, 1.0)

r["size"] = pd.cut(r["ret_factor"], [0, 1.0001, 1.02, 1.10, 1.5, np.inf],
                   labels=["no value (issue >= price)", "< 2%", "2-10%", "10-50%", "> 50% CHECK"])
print("Rights issues of our stocks:", len(r))
print(r["size"].value_counts().sort_index())
display(r.sort_values("ret_factor", ascending=False)
        [["file_symbol", "ex_date", "rights_new", "rights_held", "issue_price",
          "yesterday_close", "ret_factor", "price_jump", "subject"]].head(15))

# Factors > 1.5 are used only if the ex-day price drop confirms them (like splits in 3c.5)
r["price_confirms"] = (r["price_jump"] / r["ret_factor"]).between(0.8, 1.25) & (r["gap_days"] <= 5)
r["use"] = (r["ret_factor"] <= 1.5) | r["price_confirms"]
print("\nFactors > 1.5:", (r["ret_factor"] > 1.5).sum(), "| confirmed by the price drop:",
      ((r["ret_factor"] > 1.5) & r["price_confirms"]).sum())
rights_ok = r[r["use"]]

In [ ]:
# 3c.9 Summary and save (used by the new Step 4 returns and Step 7 share counts)

summary = {
    "split/bonus events FAILED price check (must be 0)":  int((ours["status"] == "FAILED").sum()),
    "big price moves UNKNOWN (must be 0)":                int((big["explained_by"] == "UNKNOWN").sum()),
    "manual demergers not mapped to a trading day (0)":   int(len(manual_unmapped)),
    "rights factors > 1.5 NOT confirmed by price (0)":    int((~r["use"]).sum()),
}
for k, v in summary.items():
    print(f"{k:55s} {v:4d}  {'PASS' if v == 0 else 'CHECK'}")

split_m.to_csv(INTERIM_DIR / "split_bonus_events.csv", index=False)
demerger_m.to_csv(INTERIM_DIR / "demerger_days.csv", index=False)
rights_ok.to_csv(INTERIM_DIR / "rights_events.csv", index=False)
print("\nSaved: split_bonus_events.csv (", len(split_m), ") | demerger_days.csv (", len(demerger_m),
      ") | rights_events.csv (", len(rights_ok), ")")

**Findings: Step 3c**
- NSE `prevclose` was unadjusted in 278 of 280 split/bonus events checked → the old Step 4 returns and `ca_factor` share backfill were wrong at nearly every split/bonus.
- The NSE corporate-actions file is incomplete: 14 splits/bonuses added manually (e.g. ASIANPAINT 2013 ×10, SBIN 2014 ×10, JSWSTEEL 2017 ×10) and 2 factors fixed (HINDZINC 2011, MINDACORP 2015).
- 22 demergers (e.g. RELIANCE–Jio Financial 2023, ITC–ITC Hotels 2025, TATAMOTORS 2025); before ~2019 the NSE file does not label demergers consistently, so pre-2019 ones were found from price drops.
- One-day falls of 20–37.5% not explained by an event were reviewed: apart from 2 missed bonuses (MOTHERSUMI 2015, AJANTPHARM 2022), they are real news-driven moves (e.g. ADANIENT Feb-2023, IDEA 2019–22, YESBANK 2018–19, INFY Apr-2013) and are kept.
- Rights issues: 49 for our stocks; 3 deep-discount ones (IDEA 2019, M&MFIN 2020, NCC 2014) have return factors of about 1.6, confirmed by the ex-day price drop.
- Remaining limitations: rights issues are treated as fully paid; small demergers before 2019 (one-day drop < 20%) may still be missed; several NSE demergers fall after a ~2-week trading gap, so neutralising that day also removes those two weeks' real price move.

### Step 4: Daily prices → monthly total returns (rebuilt)

**Daily return** = (close + dividend) × split factor × rights factor ÷ yesterday's close − 1
- Uses yesterday's **actual** close. NSE's `prevclose` is not adjusted for splits/bonuses in this data (Step 3c), so the old formula produced fake crashes.
- Split/bonus factor and rights factor come from Step 3c (1 on normal days).
- **Demerger days:** return set to 0 (the drop is value moved to the new company).
- Listing day and months after a month with no trades: return = NaN (unchanged rule).

**`ca_factor`** = 1 ÷ split/bonus factor, from Step 3c events. Step 7 uses it to backfill share counts.

**Checks:** known fake-crash months now normal; old vs new returns differ only in event months; remaining extreme months are real moves or data gaps.

In [ ]:
# ========================================================================================================
# STEP 4: Daily prices -> monthly TOTAL returns (+ unadjusted close for market cap)
# ========================================================================================================

In [ ]:
# 4.1 Event lookups, keyed by (file symbol, trading day) - from Step 3c

# Split/bonus: price is divided by share_mult on that day (same event under two symbols counted once)
SPLIT_MULT = (split_m.dropna(subset=["trade_date"])
                     .drop_duplicates(["file_symbol", "trade_date", "share_mult"])
                     .groupby(["file_symbol", "trade_date"])["share_mult"].prod().to_dict())

# Rights issue: return factor = cum price / theoretical ex-rights price
RIGHTS_MULT = rights_ok.groupby(["file_symbol", "trade_date"])["ret_factor"].prod().to_dict()

# Demerger: that day's drop is value moved to the new company -> daily return set to 0
DEMERGER_SET = set(zip(demerger_m["file_symbol"], demerger_m["trade_date"]))

print("Split/bonus days:", len(SPLIT_MULT), "| rights days:", len(RIGHTS_MULT),
      "| demerger days:", len(DEMERGER_SET), "| dividend events:", len(div_lookup))

In [ ]:
# 4.2 Daily prices -> monthly TOTAL returns (+ unadjusted close for market cap)
#
# Daily return = (close + dividend) x split_mult x rights_mult / yesterday's close - 1
#   - uses yesterday's ACTUAL close, never NSE's prevclose (not adjusted in this data, Step 3c)
#   - split_mult / rights_mult = 1 on normal days
#   - demerger days: return = 0

def daily_to_monthly(d, file_symbol):
    """d = daily rows of ONE stock (one price file). Returns one row per month-end."""
    d = d.copy()
    d["series"] = d["series"].astype(str).str.strip().str.upper()
    d = d[d["series"].isin(KEEP_SERIES)]
    if d.empty:
        return None

    # One row per day; if the stock switched series that day, keep EQ
    d["rank"] = d["series"].map(SERIES_RANK)
    d = d.sort_values(["date", "rank"]).drop_duplicates("date", keep="first")
    for c in ["closeprice", "turnoverinrs"]:
        d[c] = to_number(d[c])

    sym_col = "nse_symbol" if "nse_symbol" in d.columns else "symbol"
    days = list(d["date"])

    # Dividend on the ex-date: NSE file may use the symbol of that day or today's symbol
    d["div"] = [div_lookup.get((s, t), div_lookup.get((file_symbol, t), 0.0))
                for s, t in zip(d[sym_col], days)]
    d["split_mult"] = [SPLIT_MULT.get((file_symbol, t), 1.0) for t in days]
    d["rights_mult"] = [RIGHTS_MULT.get((file_symbol, t), 1.0) for t in days]
    d["is_demerger"] = [(file_symbol, t) in DEMERGER_SET for t in days]

    yesterday = d["closeprice"].shift(1)                       # first day (listing) -> NaN return
    
    # Dividend is per OLD share when it shares the ex-date with a split/bonus (identical on normal days)
    d["ret_d"] = (d["closeprice"] * d["split_mult"] + d["div"]) * d["rights_mult"] / yesterday - 1
    d.loc[d["is_demerger"], "ret_d"] = 0.0

    # Step 7 share backfill: running product halves at a 1:1 bonus
    d["ca_factor"] = 1.0 / d["split_mult"]

    d = d.set_index("date")
    m = pd.DataFrame({
        "close":         d["closeprice"].resample(MONTH_END).last(),     # UNADJUSTED, for market cap only
        "traded_value":  d["turnoverinrs"].resample(MONTH_END).sum(),    # Rs traded in the month
        "trade_days":    d["closeprice"].resample(MONTH_END).count(),
        "tt_days":       d["series"].isin(["BE", "BZ"]).resample(MONTH_END).sum(),   # trade-to-trade days
        "ret":           (1 + d["ret_d"]).resample(MONTH_END).prod() - 1,
        "ca_factor":     d["ca_factor"].resample(MONTH_END).prod(),
        "dividends":     d["div"].resample(MONTH_END).sum(),             # Rs per share, for checks
        "event_days":    ((d["split_mult"] != 1) | (d["rights_mult"] != 1)
                          | d["is_demerger"]).resample(MONTH_END).sum(),
    })

    # No trades this month or last month -> return unknown (NaN), not 0
    no_trade = (m["trade_days"] == 0) | (m["trade_days"].shift(1) == 0)
    m.loc[no_trade, "ret"] = np.nan
    return m.reset_index()


all_months = []
for f in sorted(PRICE_DIR.glob("*.csv")):
    daily = pd.read_csv(f, parse_dates=["date"])
    daily.columns = clean_cols(daily.columns)
    m = daily_to_monthly(daily, f.stem)
    if m is None:
        continue
    m["symbol"] = f.stem
    all_months.append(m)

price_data = pd.concat(all_months, ignore_index=True)
price_data = price_data[price_data["date"] <= pd.Timestamp(END_DATE)]
price_data = price_data.sort_values(["symbol", "date"]).reset_index(drop=True)

print(price_data.shape, "| stocks:", price_data["symbol"].nunique(),
      "| months with an event:", int((price_data["event_days"] > 0).sum()))

In [ ]:
# 4.3 CHECKS on the new returns

# --- Check 1: the fake crashes found in Phase 4 must be gone ---
known = [("RELIANCE", "2017-09-30"), ("BAJFINANCE", "2016-09-30"), ("BAJFINANCE", "2025-06-30"),
         ("NYKAA", "2022-11-30"), ("JUBLFOOD", "2022-04-30"), ("VBL", "2023-06-30"),
         ("VBL", "2024-09-30"), ("GICRE", "2018-07-31"), ("NESTLEIND", "2024-01-31"),
         ("TITAN", "2011-06-30"), ("JSWSTEEL", "2017-01-31"), ("ASTERDM", "2024-04-30")]
known_df = pd.DataFrame(known, columns=["symbol", "date"])
known_df["date"] = pd.to_datetime(known_df["date"])
known_df = known_df.merge(price_data[["symbol", "date", "ret", "dividends", "event_days"]],
                          on=["symbol", "date"], how="left")
print("Check 1 - known split/bonus/dividend months (returns should look normal, roughly -30% to +30%):")
display(known_df.round(3))

# --- Check 2: compare with the OLD returns (monthly_prices.csv from the previous Phase 1 run) ---
old_file = PROCESSED_DIR / "monthly_prices.csv"
if old_file.exists():
    old = pd.read_csv(old_file, parse_dates=["date"])[["symbol", "date", "ret"]]
    cmp = price_data[["symbol", "date", "ret", "event_days", "dividends"]].merge(
        old, on=["symbol", "date"], suffixes=("_new", "_old"))
    cmp["diff"] = cmp["ret_new"] - cmp["ret_old"]
    changed = cmp[cmp["diff"].abs() > 0.01]
    print(f"\nCheck 2 - months compared: {len(cmp)} | changed by > 1 pp: {len(changed)} "
          f"| of which have an event: {(changed['event_days'] > 0).sum()}")
    display(changed.reindex(changed["diff"].abs().sort_values(ascending=False).index)
                   .head(15).round(3))
else:
    print("\nCheck 2 skipped: old monthly_prices.csv not found")

# --- Check 3: extreme monthly returns left (each should be a real move or a data gap) ---
extreme = price_data[(price_data["ret"] < -0.5) | (price_data["ret"] > 1.0)]
print(f"\nCheck 3 - months with return < -50% or > +100%: {len(extreme)}")
display(extreme[["symbol", "date", "ret", "trade_days", "event_days", "dividends"]]
        .sort_values("ret").round(3))

In [ ]:
# =====================================================================
# STEP 5: Risk-free rate = 91-day T-bill (RBI auction cut-off yield)
# FF rule: Rf for month t is the T-bill rate KNOWN at the start of month t
# =====================================================================
RF_FILE = RF_DIR / "rbi_91d_tbill_cutoff.csv"

if RF_FILE.exists():
    rf = pd.read_csv(RF_FILE)
    rf["date"] = pd.to_datetime(rf["date"], dayfirst=True)
    
    RF_SOURCE = "RBI 91-day T-bill cut-off yield"
else:
    print("WARNING: RBI file not found -> using FRED interbank rate as a TEMPORARY proxy")
    
    fred = pd.read_csv("https://fred.stlouisfed.org/graph/fredgraph.csv?id=INDIR3TIB01STM")
    fred.to_csv(RF_DIR / f"fred_INDIR3TIB01STM_{RUN_DATE}.csv", index=False)   # keep the raw copy
    
    rf = fred.rename(columns={"observation_date": "date", "INDIR3TIB01STM": "yield_pct"})
    rf["date"] = pd.to_datetime(rf["date"])
    
    RF_SOURCE = "FRED INDIR3TIB01STM (3-month interbank proxy, NOT a T-bill)"

rf["yield_pct"] = pd.to_numeric(rf["yield_pct"], errors="coerce")

rf = rf.dropna(subset=["yield_pct"]).sort_values("date").set_index("date")

print("Rf source:", RF_SOURCE)

yield_month_end = rf["yield_pct"].resample(MONTH_END).last().ffill()   # latest yield at each month-end

rf_monthly = (yield_month_end / 100 / 12).shift(1)                # last month's yield = this month's Rf
rf_monthly = rf_monthly.rename("Rf_monthly").to_frame()
rf_monthly.index.name = "date"
rf_monthly.to_csv(PROCESSED_DIR / "rf_monthly.csv")

# Every return month must have an Rf
have = set(rf_monthly.dropna().index)
missing = sorted(set(price_data["date"]) - have)
print("Rf:", min(have).date(), "→", max(have).date(), "| return months without Rf:", len(missing))

In [ ]:
# =====================================================================
# STEP 6: Financial Data Pipeline
# =====================================================================

In [ ]:
import numpy as np
import re
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


def create_session():
    s = requests.Session()
    s.mount("https://", HTTPAdapter(max_retries=Retry(
        total=3, backoff_factor=2,
        status_forcelist=[429, 500, 502, 503, 504])))
    s.headers.update({
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        "Accept": "text/html,application/xhtml+xml",
    })
    return s


In [ ]:
symbols = tickers

In [ ]:
# =============================================================================
# SOURCE 2 — Screener.in annual data (HTML table scrape, NOT the JSON API)
# Gets 2014-2015 onwards reliably, annual only
# =============================================================================

from bs4 import BeautifulSoup
import re
import time

def get_html_cached(url, symbol, session, min_len=5000):
    kind = "consolidated" if "/consolidated/" in url else "standalone"
    path = SCREENER_DIR / f"{symbol}_{kind}.html"
    
    if path.exists():
        return path.read_text(encoding="utf-8")
    res = session.get(url, timeout=15)
    
    if res.status_code != 200 or len(res.text) < min_len or "No data" in res.text:
        return None                           # failures are not cached, so they retry next run
    
    path.write_text(res.text, encoding="utf-8")
    time.sleep(1.5)
    return res.text

EMPTY = {"", "-", "--", "na", "n/a", "nan"}

def to_float(text):
    """'1,234' -> 1234.0 ; '' or '-' -> None"""
    text = str(text).replace(",", "").replace("%", "").strip()
    if text.lower() in EMPTY:
        return None
    try:
        return float(text)
    except ValueError:
        return None

def read_screener_table(table):
    """Screener table -> {fiscal_year_end_date: {row_label: value}}"""
    rows = table.find_all("tr")
    headers = [c.get_text(strip=True) for c in rows[0].find_all(["th", "td"])]
    data = {}
    
    for row in rows[1:]:
        cells = [c.get_text(strip=True) for c in row.find_all(["th", "td"])]
        
        if len(cells) < 2:
            continue
        # 'Sales\xa0+' -> 'sales'
        label = cells[0].replace("\xa0", " ").rstrip("+").strip().lower()
        
        for header, value in zip(headers[1:], cells[1:]):
            m = re.search(r"([A-Za-z]{3})\s*(\d{4})", header)     # 'Mar 2024', 'Dec 2019'
            
            if m is None:
                continue                                          # skips the 'TTM' column
            fy_end = pd.to_datetime(f"1 {m.group(1)} {m.group(2)}") + pd.offsets.MonthEnd(0)
            year_dict = data.setdefault(fy_end, {})
            
            if year_dict.get(label) is None:                      # first non-empty row wins
                year_dict[label] = to_float(value)
    return data

def pick(values, names):
    """First non-empty value among the given row labels."""
    for name in names:
        if values.get(name) is not None:
            return values[name]
    return None

def parse_screener_page(html):
    """One Screener page -> one row per fiscal year (₹ crore)."""
    soup = BeautifulSoup(html, "html.parser")
    pnl_table, bal_table = None, None
    
    for sec in soup.find_all("section"):
        heading, tbl = sec.find(["h2", "h3"]), sec.find("table")
        
        if heading is None or tbl is None:
            continue
        title = heading.get_text(strip=True).lower()
        
        if title == "profit & loss" and pnl_table is None:
            pnl_table = tbl
        elif title == "balance sheet" and bal_table is None:
            bal_table = tbl
    
    if pnl_table is None or bal_table is None:
        return None

    pnl = read_screener_table(pnl_table)
    bal = read_screener_table(bal_table)

    records = []
    
    for fy_end in sorted(pnl):
        p = pnl[fy_end]
        b = bal.get(fy_end, {})               # SAME fiscal year only. Missing -> empty, never another year
        cap = pick(b, ["equity capital", "share capital"])
        res = pick(b, ["reserves"])
        book = cap + res if (cap is not None and res is not None) else None
        records.append({
            "fy_end":           fy_end,
            "year":             fy_end.year,
            "quarter":          "FY",
            "sales":            pick(p, ["sales", "revenue"]),
            "operating_profit": pick(p, ["operating profit"]),   # EBITDA before other income
            "other_income":     pick(p, ["other income"]),
            "interest_expense": pick(p, ["interest"]),
            "depreciation":     pick(p, ["depreciation"]),
            "net_income":       pick(p, ["net profit"]),
            "total_assets":     pick(b, ["total assets"]),
            "book_value":       book,                           # equity capital + reserves
        })
    return pd.DataFrame(records)

In [ ]:
# =============================================================================
# MASTER PIPELINE: consolidated preferred, standalone fills the missing years
# =============================================================================

def run_pipeline(symbol, session):
    frames = []
    
    for basis, url in [
        ("consolidated", f"https://www.screener.in/company/{symbol}/consolidated/?view=15years"),
        ("standalone",   f"https://www.screener.in/company/{symbol}/?view=15years"),
    ]:
        try:
            html = get_html_cached(url, symbol, session)
        
        except requests.RequestException as e:
            print(f"  [{basis}] network error: {e}", end="")
            continue
        
        if html is None:
            continue
        df = parse_screener_page(html)
        
        if df is not None and not df.empty:
            df["basis"] = basis
            frames.append(df)

    if not frames:
        return None, "screener_failed"              # no crash when both pages fail

    both = pd.concat(frames, ignore_index=True)
    both = both[both["total_assets"].notna()]       # a year without a balance sheet is unusable
    
    # Same fiscal year on both pages -> keep consolidated (the Compustat convention)
    both["basis_rank"] = both["basis"].map({"consolidated": 0, "standalone": 1})
    both = (both.sort_values(["fy_end", "basis_rank"])
                .drop_duplicates("fy_end", keep="first")
                .drop(columns="basis_rank"))
    both.insert(0, "symbol", symbol)
    print(f"  {len(both)} yrs", end="")
    
    return both.sort_values("fy_end").reset_index(drop=True), None

In [ ]:
# =============================================================================
# EXECUTION: annual fundamentals, all values in ₹ crore (as on Screener)
# =============================================================================

session = create_session()
all_data, failures = [], {}

for i, sym in enumerate(symbols, 1):
    print(f"\n[{i}/{len(symbols)}] {sym}", end="  ")
    df, err = run_pipeline(sym, session)
    
    if df is None or df.empty:
        failures[sym] = err or "no_valid_years"
        print(f"  → FAILED: {failures[sym]}", end="")
        continue
    
    all_data.append(df)
    
    if i % 50 == 0:
        pd.concat(all_data).to_csv(FIN_BACKUPS_DIR / f"fundamentals_backup_{i}.csv", index=False)

final_df = pd.concat(all_data, ignore_index=True)

pd.DataFrame(list(failures.items()), columns=["symbol", "reason"]).to_csv(
    FIN_BACKUPS_DIR / "failures.csv", index=False)

# FF5 operating profitability: (Operating profit − Interest) / Book equity, same fiscal year
ok = (final_df["operating_profit"].notna() & final_df["interest_expense"].notna()
      & (final_df["book_value"] > 0))

final_df["operating_profitability"] = np.where(
    ok, (final_df["operating_profit"] - final_df["interest_expense"]) / final_df["book_value"], np.nan)

# Flags for Phase 2
final_df["is_financial"] = final_df["symbol"].isin(FIN_SYMBOLS)
final_df["months_since_prev_fy"] = (final_df.groupby("symbol")["fy_end"].diff().dt.days / 30.44).round()
final_df["prev_basis"] = final_df.groupby("symbol")["basis"].shift(1)

# Data-error check: tiny book equity vs assets (e.g. MSUMI: BE ₹0.1 cr -> OP = 5,460)
final_df["check_me"] = (final_df["book_value"] > 0) & (final_df["book_value"] < 0.01 * final_df["total_assets"])
print(final_df.loc[final_df["check_me"],
      ["symbol", "fy_end", "book_value", "total_assets", "operating_profitability"]])

print(f"\nSymbols : {final_df['symbol'].nunique()}")
print(f"Rows    : {len(final_df)}")
print(f"FY range: {final_df['fy_end'].min().date()} → {final_df['fy_end'].max().date()}")
print(f"Failures: {len(failures)}")
print(final_df["basis"].value_counts())
print("Odd-length fiscal years (e.g. 15 months):", (final_df["months_since_prev_fy"] > 13).sum())

**Data limitations (fundamentals)**
- Screener gives 12 annual columns: FY2015 onward. B/M and OP start FY2015; CMA (needs 2 years) starts FY2016.
  → First full 5-factor year is Jul-2016; about 10 years of factor returns, which is short for significant premia.
- Indian GAAP → Ind AS change around FY2016–FY2018 changes book equity. Revaluation reserves cannot be removed with Screener data.
- Units: every accounting value is ₹ crore, as published by Screener. No conversion anywhere.
- Financial firms (NSE industry "Financial Services") are excluded from the SMB/HML/RMW/CMA sorts; MKT includes them.
- Basis: consolidated where available, standalone otherwise (column `basis`).

In [ ]:
# =====================================================================
# STEP 6b: Availability rule (replaces get_date + monthly ffill + shift(3))
# SEBI LODR Reg 33: audited annual results within 60 days of FY end
# -> 3-month lag is safe: FY Mar-2017 is usable from 30-Jun-2017.
# =====================================================================

final_df["available_date"] = final_df["fy_end"] + pd.offsets.MonthEnd(LAG_MONTHS)

def fundamentals_as_of(fund, formation_date):
    """One row per stock: the latest fiscal year that was PUBLIC on formation_date.
    Only fiscal years ending in the 12 months up to (formation_date − lag) count.
    This is the Indian version of FF's 'fiscal year ending in calendar year t−1'."""
    
    F = pd.Timestamp(formation_date)
    oldest_ok = F - pd.offsets.MonthEnd(LAG_MONTHS + 12)
    
    usable = fund[(fund["available_date"] <= F) & (fund["fy_end"] > oldest_ok)]
    
    return usable.sort_values("fy_end").groupby("symbol").tail(1)

# Test: a June-2017 formation must pick FY Mar-2017, never Mar-2016
print(fundamentals_as_of(final_df, "2017-06-30")["fy_end"].value_counts().head())

In [ ]:
# import sys
# !{sys.executable} -m pip install plotly

In [ ]:
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'browser'   # safest across environments

# Step 1: Count + sort years
year_counts = (
    final_df['year']
    .astype(int)
    .value_counts()
    .sort_index()
    .reset_index()
)

year_counts.columns = ['year', 'count']

# Step 2: Plot
fig = px.bar(
    year_counts,
    x='year',
    y='count',
    title='Number of Data Points per Year',
    labels={'year': 'Year', 'count': 'Data Availability: Number of stocks out of 500'}
)

# Optional styling
fig.update_layout(
    xaxis=dict(dtick=1),   # show every year
    template='plotly_white'
)

fig.show()

In [ ]:
# All accounting values stay in ₹ crore, exactly as on Screener. Do NOT convert.
# (Market cap will be converted to crore in Step 7: close × shares / 1e7)

In [ ]:
final_df[['operating_profit', 
        'interest_expense', 
        'depreciation', 
        'net_income', 
        'total_assets', 
        'operating_profitability',
        'book_value']].describe().transpose()


In [ ]:
final_df.shape

In [ ]:
final_df.head()

In [ ]:
final_df.to_csv(PROCESSED_DIR / "final_fin_data.csv", index=False)

In [ ]:
price_data.head()

In [ ]:
price_data.columns

In [ ]:
price_data.describe().transpose()

In [ ]:
# =====================================================================
# STEP 7: Factor inputs (one row per stock per formation year)
# =====================================================================
import yfinance as yf

FORMATION_MONTH = 6       # portfolios formed at end of June, held July -> next June

In [ ]:
# =====================================================================
# 7.1 Shares outstanding (yfinance), saved to disk so we download only once
# =====================================================================
SHARES_DIR = RAW_DIR / "shares_yf"
SHARES_DIR.mkdir(parents=True, exist_ok=True)

def get_shares(symbol):
    """Daily share counts for one stock, or None."""
    path = SHARES_DIR / f"{symbol}.csv"

    # Already downloaded before -> read from disk
    if path.exists():
        return pd.read_csv(path, parse_dates=["date"])

    try:
        s = yf.Ticker(f"{symbol}.NS").get_shares_full(start="2011-01-01")
    except Exception as e:
        print(symbol, "yfinance error:", e)
        return None

    if s is None or len(s) == 0:
        return None

    # Remove the time zone but keep the local date
    dates = pd.to_datetime(s.index)
    if dates.tz is not None:
        dates = dates.tz_localize(None)

    df = pd.DataFrame({"date": dates, "shares": s.values})
    df.to_csv(path, index=False)
    time.sleep(0.5)
    return df


In [ ]:
# One value per month: the median share count reported in that month
monthly_shares = []

for sym in price_data["symbol"].unique():
    daily = get_shares(sym)
    if daily is None:
        continue
    daily = daily.set_index("date")
    one = daily["shares"].resample(MONTH_END).median().reset_index()
    one = one.rename(columns={"shares": "shares_yf"})
    one["symbol"] = sym
    monthly_shares.append(one)

shares_yf = pd.concat(monthly_shares, ignore_index=True)
shares_yf = shares_yf.dropna(subset=["shares_yf"])
print("Stocks with yfinance shares:", shares_yf["symbol"].nunique())

In [ ]:
# =====================================================================
# 7.2 Shares for every month + market cap (₹ crore)
# =====================================================================

price_data = price_data.merge(shares_yf, on=["symbol", "date"], how="left", validate="1:1")

# Carry a yfinance value forward at most 3 months (yfinance has gaps)
price_data = price_data.sort_values(["symbol", "date"]).reset_index(drop=True)
# Carry a yfinance value forward at most 3 months (yfinance has gaps)
price_data["shares_yf"] = price_data.groupby("symbol")["shares_yf"].ffill(limit=3)


In [ ]:
def fill_shares(one):
    """one = monthly rows of ONE stock, sorted by date."""
    one = one.copy()

    # Running product of split/bonus factors (halves at a 1:1 bonus), from Step 3c events
    one["split_index"] = one["ca_factor"].cumprod()

    # Shares in split-free units stay constant through splits/bonuses, so gaps can be filled safely:
    # forward from the last yfinance value, and backward before the first one
    norm = one["shares_yf"] * one["split_index"]
    one["shares"] = norm.ffill().bfill() / one["split_index"]

    one["shares_source"] = np.select(
        [one["shares_yf"].notna(), norm.ffill().notna(), norm.bfill().notna()],
        ["yfinance", "carried forward", "backfilled"], default="missing")
    return one

In [ ]:
filled = []
for sym, one in price_data.groupby("symbol"):
    filled.append(fill_shares(one))
price_data = pd.concat(filled, ignore_index=True)

# Market cap in ₹ crore: UNADJUSTED close × total shares (FF uses total shares, not free float)
price_data["me"] = price_data["close"] * price_data["shares"] / 1e7

# Last month's market cap = the weight for this month's value-weighted return
price_data["me_lag"] = price_data.groupby("symbol")["me"].shift(1)

print(price_data["shares_source"].value_counts())

In [ ]:
# =====================================================================
# 7.3 Per stock, per fiscal year: asset growth, ME at FY end, B/M
# =====================================================================

fund = final_df.sort_values(["symbol", "fy_end"]).reset_index(drop=True)

# --- CMA input: total-asset growth from FY t-1 to FY t ---
fund["prev_total_assets"] = fund.groupby("symbol")["total_assets"].shift(1)
fund["asset_growth"] = fund["total_assets"] / fund["prev_total_assets"] - 1


In [ ]:
# Only valid if the two years are exactly 12 months apart and on the same basis
not_12_months = fund["months_since_prev_fy"] != 12
basis_changed = fund["basis"] != fund["prev_basis"]
bad_assets = fund["prev_total_assets"] <= 0
fund.loc[not_12_months | basis_changed | bad_assets, "asset_growth"] = np.nan


In [ ]:
# --- Market cap at the firm's own fiscal year-end (India's version of FF's December ME) ---
me_fy = price_data[["symbol", "date", "me"]]
me_fy = me_fy.rename(columns={"date": "fy_end", "me": "me_fy_end"})
fund = fund.merge(me_fy, on=["symbol", "fy_end"], how="left", validate="1:1")


In [ ]:
# --- HML input: B/M = book equity / ME, both at the same fiscal year-end, both ₹ crore ---
fund["bm"] = fund["book_value"] / fund["me_fy_end"]
fund.loc[fund["book_value"] <= 0, "bm"] = np.nan      # FF: negative book equity -> excluded
fund.loc[fund["me_fy_end"] <= 0, "bm"] = np.nan

In [ ]:
# =====================================================================
# 7.4 Characteristics at each June formation date
# =====================================================================

first_year = fund["fy_end"].min().year
snapshots = []

for year in range(first_year, END_DATE.year + 1):
    F = pd.Timestamp(year=year, month=FORMATION_MONTH, day=1) + pd.offsets.MonthEnd(0)
    if F > pd.Timestamp(END_DATE):
        break
    snap = fundamentals_as_of(fund, F).copy()     # latest fiscal year that was public on F
    snap["formation_date"] = F
    snapshots.append(snap)


In [ ]:
chars = pd.concat(snapshots, ignore_index=True)

# Add values measured AT the formation date: size (June ME) and the eligibility helpers
at_F = price_data[["symbol", "date", "me", "shares_source",
                   "months_traded", "tv_median_12m", "trade_days_12m"]]
at_F = at_F.rename(columns={"date": "formation_date", "me": "me_formation"})
chars = chars.merge(at_F, on=["symbol", "formation_date"], how="left", validate="1:1")

print(chars.shape)

In [ ]:
# =====================================================================
# 7.5 Monthly panel: returns, Rf, weights, and which formation each month belongs to
# =====================================================================

panel = price_data[["symbol", "date", "ret", "close", "shares", "me", "me_lag",
                    "shares_source", "trade_days", "tt_days"]].copy()

panel = panel.merge(rf_monthly.reset_index(), on="date", how="left", validate="m:1")
panel["excess_ret"] = panel["ret"] - panel["Rf_monthly"]
panel["is_financial"] = panel["symbol"].isin(FIN_SYMBOLS)


In [ ]:
def formation_date_for(month_end):
    """Jul-2017 ... Jun-2018 -> 2017-06-30 (the June that formed the portfolio)."""
    if month_end.month > FORMATION_MONTH:
        year = month_end.year
    else:
        year = month_end.year - 1
    return pd.Timestamp(year=year, month=FORMATION_MONTH, day=1) + pd.offsets.MonthEnd(0)


panel["formation_date"] = panel["date"].map(formation_date_for)
print(panel.shape)

In [ ]:
# =====================================================================
# 7.6 Checks
# =====================================================================

In [ ]:
# 1) How many stocks have each characteristic, per formation date
summary = chars.groupby("formation_date").agg(
    stocks=("symbol", "count"),
    size=("me_formation", "count"),
    bm=("bm", "count"),
    op=("operating_profitability", "count"),
    inv=("asset_growth", "count"),
)
print(summary)


In [ ]:
# 2) Share-count source: how much is backfilled
print(chars.groupby("formation_date")["shares_source"].value_counts().unstack())


In [ ]:
# 3) Share counts that jump >50% in a month with no split/bonus = bad yfinance data
price_data["shares_change"] = price_data.groupby("symbol")["shares"].pct_change(fill_method=None)
jumps = price_data[(price_data["shares_change"].abs() > 0.5) & (price_data["ca_factor"] == 1)]
print("Suspicious share jumps:", len(jumps))
print(jumps[["symbol", "date", "shares", "shares_change"]].head(20))


In [ ]:
# 4) OP at formation dates only (each firm-year counted once), non-financials
non_fin = chars[~chars["is_financial"]]
print(non_fin["operating_profitability"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))


In [ ]:
# 5) Tiny book equity flagged in Step 6 (MSUMI-type errors)
print(chars.loc[chars["check_me"], ["symbol", "formation_date", "book_value", "operating_profitability"]])

In [ ]:
# =====================================================================
# 7.7 Save Phase 1 outputs
# =====================================================================

chars.to_csv(PROCESSED_DIR / "characteristics_annual.csv", index=False)
panel.to_csv(PROCESSED_DIR / "monthly_panel.csv", index=False)
print("Saved: characteristics_annual.csv  (one row per stock per June formation)")
print("Saved: monthly_panel.csv           (one row per stock per month)")

## Phase 1 Summary: Data Ingestion & Cleaning

| Output | Contents |
|---|---|
| `monthly_panel.csv` | Stock × month: total return (dividends, split-adjusted), Rf, excess return, ME (₹ cr), ME of previous month (weights), formation_date |
| `characteristics_annual.csv` | Stock × June formation: ME in June (size), B/M, OP, asset growth, eligibility helpers, flags |
| `rf_monthly.csv` | 91-day T-bill yield / 12, known at start of month |

**Definitions (FF5)**
- Size = ME at end of June T. ME = unadjusted close × total shares (not free float), ₹ crore.
- B/M = book equity (equity capital + reserves) / ME, both at the firm's fiscal year-end.
- OP = (operating profit − interest) / book equity, same fiscal year.
- Investment = total assets FY t / total assets FY t−1 − 1 (only if 12 months apart, same basis).
- One fiscal year per stock per formation: latest FY public by June T (3-month lag, SEBI LODR Reg 33).

**Phase 2 rules**
- 2×3 sorts: size at the median, B/M, OP, investment at 30th/70th percentiles. SMB = average of the three size spreads.
- Rebalance each June; hold July–June; value-weight with previous month's ME.
- MKT uses all stocks. The four characteristic sorts exclude financials. BE > 0 required for the HML and RMW sorts.
- Eligibility at formation: months_traded, tv_median_12m, trade_days_12m (thresholds set in Phase 2).

**Limitations**
- Fundamentals start FY2015 (Screener), so first full 5-factor year is Jul-2016.
- Universe = today's Nifty 500 (survivorship) until the bhavcopy universe replaces it.
- Shares before 2017 are backfilled from the first yfinance value using splits/bonuses only.
- Rf from FRED interbank rate until the RBI T-bill file is added (if applicable).

In [12]:
import sys
!{sys.executable} -m pip install xlrd

In [14]:
# Inspect the NSE market-cap ranking files and the symbol-change list
# (.xls needs: %pip install xlrd)
folder = RAW_DIR / "universe" / "nse_mcap"
for f in sorted(folder.glob("*")):
    print("\n=====", f.name)
    if f.suffix == ".csv":
        df = pd.read_csv(f, encoding="latin-1")
        print(df.shape); print(df.head(3).to_string())
    else:
        sheets = pd.read_excel(f, sheet_name=None, header=None, nrows=8)   # header row position unknown
        for name, df in sheets.items():
            print("--- sheet:", name)
            print(df.to_string())


===== Average_MCAP_July2025ToDecember2025_20260102201101.xlsx
--- sheet: Sheet1
      0           1                                  2                                                                                      3
0  Rank      Symbol                       Company Name  Average market capitalisation from July  01, 2025 to December 31, 2025 (Rs. In lakhs)
1     1    RELIANCE        Reliance Industries Limited                                                                       197078768.997765
2     2    HDFCBANK                  HDFC Bank Limited                                                                       151558090.449832
3     3  BHARTIARTL              Bharti Airtel Limited                                                                       113625128.614847
4     4         TCS  Tata Consultancy Services Limited                                                                       112955514.997768
5     5   ICICIBANK                 ICICI Bank Limited             

In [16]:
print(SCREENER_DIR.resolve())
print((SCREENER_DIR / "HDFCBANK_consolidated.html").exists(),
      (SCREENER_DIR / "TCS_standalone.html").exists())

C:\Users\jayan\Study Materials\Fama-French_model\data\raw\screener_html\2026-10-14
True True
